In [4]:
import torch
from torch import nn
import torch.nn.functional as F

torch.manual_seed(42)

vocab = {
    "<pad>": 0,
    "i": 1,
    "like": 2,
    "pytorch": 3,
    "love": 4,
}

tokens = ["i", "like", "pytorch", "i"]

token_ids = torch.tensor(
    [vocab[token] for token in tokens],
    dtype=torch.long
)

print("Tokens:", tokens)
print("Token IDs:", token_ids)

Tokens: ['i', 'like', 'pytorch', 'i']
Token IDs: tensor([1, 2, 3, 1])


In [5]:
vocab_size = len(vocab)
embedding_dim = 4

embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embedding_dim
)

vectors = embedding(token_ids)

# .weight 就是词 → 向量的映射表本体，Embedding 的查表操作就是读它
print("Embedding weight shape:", embedding.weight.shape)
print("Output shape:", vectors.shape)
print("Output vectors:\n", vectors)

Embedding weight shape: torch.Size([5, 4])
Output shape: torch.Size([4, 4])
Output vectors:
 tensor([[-0.7581,  1.0783,  0.8008,  1.6806],
        [ 0.3559, -0.6866, -0.4934,  0.2415],
        [-0.2316,  0.0418, -0.2516,  0.8599],
        [-0.7581,  1.0783,  0.8008,  1.6806]], grad_fn=<EmbeddingBackward0>)


In [6]:
lookup_vectors = embedding.weight[token_ids]

print("Embedding output:\n", vectors)
print("Direct lookup:\n", lookup_vectors)

assert torch.allclose(vectors, lookup_vectors)

one_hot = F.one_hot(
    token_ids,
    num_classes=vocab_size
).float()

one_hot_vectors = one_hot @ embedding.weight

print("One-hot shape:", one_hot.shape)
print("One-hot multiplication result:\n", one_hot_vectors)

assert torch.allclose(vectors, one_hot_vectors)

Embedding output:
 tensor([[-0.7581,  1.0783,  0.8008,  1.6806],
        [ 0.3559, -0.6866, -0.4934,  0.2415],
        [-0.2316,  0.0418, -0.2516,  0.8599],
        [-0.7581,  1.0783,  0.8008,  1.6806]], grad_fn=<EmbeddingBackward0>)
Direct lookup:
 tensor([[-0.7581,  1.0783,  0.8008,  1.6806],
        [ 0.3559, -0.6866, -0.4934,  0.2415],
        [-0.2316,  0.0418, -0.2516,  0.8599],
        [-0.7581,  1.0783,  0.8008,  1.6806]], grad_fn=<IndexBackward0>)
One-hot shape: torch.Size([4, 5])
One-hot multiplication result:
 tensor([[-0.7581,  1.0783,  0.8008,  1.6806],
        [ 0.3559, -0.6866, -0.4934,  0.2415],
        [-0.2316,  0.0418, -0.2516,  0.8599],
        [-0.7581,  1.0783,  0.8008,  1.6806]], grad_fn=<MmBackward0>)


In [7]:
torch.manual_seed(42)

embedding = nn.Embedding(
    num_embeddings=5,
    embedding_dim=3
)

token_ids = torch.tensor([1, 3, 1], dtype=torch.long)

vectors = embedding(token_ids)

loss = vectors.sum()
loss.backward()

print("Embedding gradients:\n", embedding.weight.grad)

grad = embedding.weight.grad

assert torch.all(grad[1] == 2)
assert torch.all(grad[3] == 1)
assert torch.all(grad[0] == 0)
assert torch.all(grad[2] == 0)
assert torch.all(grad[4] == 0)

Embedding gradients:
 tensor([[0., 0., 0.],
        [2., 2., 2.],
        [0., 0., 0.],
        [1., 1., 1.],
        [0., 0., 0.]])


In [8]:
import torch
from torch import nn

torch.manual_seed(42)

embedding = nn.Embedding(
    num_embeddings=5,
    embedding_dim=3,
    padding_idx=1
)

token_ids = torch.tensor([1, 3, 1], dtype=torch.long)

vectors = embedding(token_ids)

loss = vectors.sum()
loss.backward()

print("Embedding gradients:\n", embedding.weight.grad)

grad = embedding.weight.grad

assert torch.all(grad[1] == 0)
assert torch.all(grad[3] == 1)
assert torch.all(grad[0] == 0)
assert torch.all(grad[2] == 0)
assert torch.all(grad[4] == 0)

Embedding gradients:
 tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [1., 1., 1.],
        [0., 0., 0.]])
